# 15 - Codility 4: binary search, caterpillar, greedy, dynamic programming

**Source:** *Codility Lessons* chapters 14-17. Practice: `min_max_division`, `abs_distinct`, `count_triangles`,
`tie_ropes`, `max_nonoverlapping_segments`, `coin_change_min`, `frog_ways` in
`codility/practice_04_numbers_search_greedy_dp.py`.

**Bugs / traps found in the PDF code (fixed below):**
* `mid = (beg + end) / 2` -> float in Python 3 -> `//`.
* `dynamic_coin_changing` uses an undefined `MAX_INT` -> `math.inf` (or a large int).
* `greedyCoinChanging` is only correct for some coin systems (the PDF says so; we *demonstrate* it).
* `caterpillarMethod` when `front == back` and `A[back] > s`: `total -= A[back]` is executed although the element
  was never added (negative total). Guard it.
* `check(A, k)` in *boards* is named like the function `boards` - fine, but its parameter `k` is the board *size*.

In [1]:
import math
import random
rng = random.Random(8)

## 1. Binary search
Search a **sorted** sequence (or any *monotone predicate*) by halving the interval: **O(log n)**.

Template - "largest index with `A[i] <= x`" (the PDF's version):

In [2]:
def binary_search_le(A, x):
    beg, end, result = 0, len(A) - 1, -1
    while beg <= end:
        mid = (beg + end) // 2              # `/` in the PDF would give a float index
        if A[mid] <= x:
            beg, result = mid + 1, mid      # candidate found; try to the right
        else:
            end = mid - 1
    return result

import bisect
A = [12, 15, 15, 19, 24, 31, 53, 59, 60]
assert binary_search_le(A, 31) == 5 and binary_search_le(A, 11) == -1 and binary_search_le(A, 100) == 8
for _ in range(1000):
    A = sorted(rng.randint(0, 20) for _ in range(rng.randint(0, 10)))
    x = rng.randint(-1, 21)
    assert binary_search_le(A, x) == bisect.bisect_right(A, x) - 1
print("matches bisect.bisect_right(A, x) - 1")

matches bisect.bisect_right(A, x) - 1


The standard library has it: `bisect_left(A, x)` = first index with `A[i] >= x`; `bisect_right(A, x)` = first index with
`A[i] > x`; `insort`. Use them in real code; know the loop for "binary search on the answer".

**Overflow note:** in C/Java `(beg + end) / 2` can overflow, use `beg + (end - beg) / 2`. Python ints don't overflow.

### Binary search on the result
If "size `x` is enough" implies "size `x + 1` is enough" (monotone), binary-search the **smallest** enough `x`; each guess
is verified by a (usually greedy) linear check -> O(n log n).

In [3]:
def check(A, size):
    """Minimum number of boards of length `size` needed to cover every hole (1) in A - greedy, left to right."""
    boards, last = 0, -1
    for i, hole in enumerate(A):
        if hole == 1 and last < i:
            boards += 1
            last = i + size - 1
    return boards

def boards(A, k):
    beg, end, result = 1, len(A), -1
    while beg <= end:
        mid = (beg + end) // 2
        if check(A, mid) <= k:
            end, result = mid - 1, mid
        else:
            beg = mid + 1
    return result

def boards_linear(A, k):
    return next((s for s in range(1, len(A) + 1) if check(A, s) <= k), -1)

assert boards([1, 0, 0, 1, 1, 0, 1], 2) == 4
for _ in range(500):
    A = [rng.randint(0, 1) for _ in range(rng.randint(1, 12))]
    k = rng.randint(1, 4)
    assert boards(A, k) == boards_linear(A, k)
print("boards() agrees with a linear scan")

boards() agrees with a linear scan


Recognise the pattern: *minimise the maximum* / *maximise the minimum* / "smallest X such that feasible(X)" -> binary
search on the answer (*MinMaxDivision*, *NailingPlanks*, *Aggressive cows*, *Koko eating bananas*...). Tips: pick tight
bounds (`lo = max(A)`, `hi = sum(A)` for splitting arrays); make sure `feasible` is truly monotone; use `//`.

## 2. Caterpillar method (two pointers / sliding window)
Two indices `back <= front` crawl forward; each only moves right, so the loop is **O(n)** even with a nested `while`
(amortised analysis). Works when elements are non-negative so that growing the window increases the sum.

In [4]:
def caterpillar_pdf(A, s):
    """The PDF's version translated 1:1 (can drive `total` negative)."""
    n = len(A)
    front, total = 0, 0
    for back in range(n):
        while front < n and total + A[front] <= s:
            total += A[front]
            front += 1
        if total == s:
            return True
        total -= A[back]
    return False

def caterpillar(A, s):
    n = len(A)
    front, total = 0, 0
    for back in range(n):
        while front < n and total + A[front] <= s:
            total += A[front]
            front += 1
        if total == s:
            return True
        if front > back:                    # only remove what has actually been added
            total -= A[back]
        else:                               # window is empty (A[back] > s): skip past this element
            front = back + 1
    return False

def contiguous_sum_brute(A, s):
    return any(sum(A[p:q + 1]) == s for p in range(len(A)) for q in range(p, len(A)))

assert caterpillar([6, 2, 7, 4, 1, 3, 6], 12) and not caterpillar([6, 2, 7, 4, 1, 3, 6], 5.5)
mismatch_pdf = None
for _ in range(20000):
    A = [rng.randint(1, 9) for _ in range(rng.randint(1, 7))]
    s = rng.randint(1, 15)
    assert caterpillar(A, s) == contiguous_sum_brute(A, s), (A, s)
    if mismatch_pdf is None and caterpillar_pdf(A, s) != contiguous_sum_brute(A, s):
        mismatch_pdf = (A, s)
print("fixed caterpillar == brute force; first counter-example for the PDF version:", mismatch_pdf)

fixed caterpillar == brute force; first counter-example for the PDF version: None


### Counting triangles (PDF 15.2) - O(n^2)
Sort. For each `x` and `y > x`, the largest `z` with `A[x] + A[y] > A[z]` can only increase when `y` increases, so
`z` never moves back. Every `k` in `(y, z)` also works -> add `z - y - 1` at once.

In [5]:
def triangles_sorted(A):                    # A must be sorted
    n, result = len(A), 0
    for x in range(n):
        z = x + 2
        for y in range(x + 1, n):
            while z < n and A[x] + A[y] > A[z]:
                z += 1
            result += max(0, z - y - 1)     # max(0, .) when z < y + 1 (the PDF can go negative if z lags behind y)
    return result

import itertools
def triangles_brute(A):
    return sum(1 for a, b, c in itertools.combinations(A, 3) if a + b > c and a + c > b and b + c > a)

for _ in range(500):
    A = sorted(rng.randint(1, 15) for _ in range(rng.randint(0, 9)))
    assert triangles_sorted(A) == triangles_brute(A), A
print("triangles ok")

triangles ok


Related two-pointer tasks: *abs distinct* (walk in from both ends), *two-sum in a sorted array*, longest substring
without repeats (window + set/dict), minimum window with sum >= s, merge of sorted lists, *count pairs with difference d*.

## 3. Greedy algorithms
Make the **locally best** choice at every step. Correct only if you can prove it, typically by an **exchange argument**:
"if an optimal solution does not make this choice, we can modify it to do so without making it worse".

### Coin changing: when greedy works and when it fails

In [6]:
def greedy_coin_changing(M, k):
    result = []
    for coin in sorted(M, reverse=True):
        result.append((coin, k // coin))
        k %= coin
    return result, k                          # k is what is left unpaid (non-zero when impossible)

print(greedy_coin_changing([1, 2, 5], 11))   # [(5, 2), (2, 0), (1, 1)] -> 3 coins, optimal
print(greedy_coin_changing([1, 3, 4], 6))    # [(4, 1), (3, 0), (1, 2)] -> 3 coins, but 3 + 3 uses only 2!

([(5, 2), (2, 0), (1, 1)], 0)
([(4, 1), (3, 0), (1, 2)], 0)


### Canoeists (PDF 16.3)
Seat the heaviest with the lightest if they fit together, otherwise the heaviest sits alone: **two pointers, O(n)**.
(Solution A in the PDF is more complicated; B is the one to remember.)

In [7]:
def canoeist_b(W, k):                         # W sorted ascending, each w <= k
    canoes, j, i = 0, 0, len(W) - 1
    while i >= j:
        if W[i] + W[j] <= k:
            j += 1
        canoes += 1
        i -= 1
    return canoes

def canoeist_brute(W, k):
    """Minimum number of canoes (each holds at most two people with total weight <= k) by bitmask DP."""
    from functools import lru_cache
    n = len(W)

    @lru_cache(None)
    def best(mask):
        if mask == (1 << n) - 1:
            return 0
        i = next(b for b in range(n) if not mask & (1 << b))
        res = 1 + best(mask | 1 << i)                         # i goes alone
        for j in range(i + 1, n):
            if not mask & (1 << j) and W[i] + W[j] <= k:
                res = min(res, 1 + best(mask | 1 << i | 1 << j))
        return res

    return best(0)

for _ in range(500):
    k = rng.randint(5, 20)
    W = sorted(rng.randint(1, k) for _ in range(rng.randint(1, 8)))
    assert canoeist_b(W, k) == canoeist_brute(tuple(W), k), (W, k)
print("canoeists ok")

canoeists ok


### Classic greedy problems (-> `practice_04`)
* **Activity selection / max non-overlapping segments:** sort by *end*, take every segment starting after the last taken end.
* **Tie ropes:** accumulate until the sum reaches `K`, then cut. * **Scheduling** with deadlines, **Huffman coding** (heap),
  **fractional knapsack**, **minimum platforms/rooms** (sort events or heap).
* Typical tools: `sort`, `heapq` (priority queue), two pointers.
* If you cannot prove the greedy choice, test it against brute force or think DP.

## 4. Dynamic programming
Solve a problem from solutions to **overlapping sub-problems**; store them (table/array) to avoid recomputation.
Recipe: (1) define the state `dp[...]` in words; (2) base cases; (3) recurrence; (4) order of evaluation; (5) answer
cell; (6) complexity and possible memory reduction (keep only the previous row).

### Coin changing (minimum number of coins)
`dp[j]` = minimum number of coins for amount `j` using the denominations considered so far.

In [8]:
INF = math.inf                                # the PDF's undefined MAX_INT

def dynamic_coin_changing_2d(C, k):
    n = len(C)
    dp = [[0] * (k + 1) for _ in range(n + 1)]
    dp[0] = [0] + [INF] * k
    for i in range(1, n + 1):
        for j in range(C[i - 1]):
            dp[i][j] = dp[i - 1][j]
        for j in range(C[i - 1], k + 1):
            dp[i][j] = min(dp[i][j - C[i - 1]] + 1, dp[i - 1][j])
    return dp

def dynamic_coin_changing(C, k):               # memory reduced to O(k)
    dp = [0] + [INF] * k
    for c in C:
        for j in range(c, k + 1):
            dp[j] = min(dp[j - c] + 1, dp[j])
    return dp

table = dynamic_coin_changing_2d([1, 3, 4], 6)
for row in table:
    print(row)
assert dynamic_coin_changing([1, 3, 4], 6) == table[-1] == [0, 1, 2, 1, 1, 2, 2]
assert dynamic_coin_changing([5, 10], 3)[3] == INF

[0, inf, inf, inf, inf, inf, inf]
[0, 1, 2, 3, 4, 5, 6]
[0, 1, 2, 1, 2, 3, 2]
[0, 1, 2, 1, 1, 2, 2]


### Frog jumps: counting paths modulo q (PDF 17.2)
`dp[j]` = number of ways to land exactly on `j` = sum of `dp[j - s]` over the allowed jumps `s`. Order matters
(1+2 and 2+1 are different), which is why the *position* loop is outermost. If you swapped the loops
(coin loop outside) you would count *combinations* instead - compare `frog_ways` with `coin_ways` below.

In [9]:
def frog(S, k, q):
    dp = [1] + [0] * k
    for j in range(1, k + 1):
        for s in S:
            if s <= j:
                dp[j] = (dp[j] + dp[j - s]) % q
    return dp[k]

def coin_ways(C, k):                           # number of COMBINATIONS (order does not matter)
    dp = [1] + [0] * k
    for c in C:
        for j in range(c, k + 1):
            dp[j] += dp[j - c]
    return dp[k]

print(frog([1, 2], 4, 1000), coin_ways([1, 2], 4))    # 5 sequences vs 3 combinations: 1111, 112, 22
assert frog([1, 2], 10, 10**9) == 89

5 3


### Other patterns you should be able to write from memory

In [10]:
# 0/1 knapsack - O(n * W), iterate capacities DOWNWARDS so each item is used once
def knapsack(weights, values, W):
    dp = [0] * (W + 1)
    for w, v in zip(weights, values):
        for cap in range(W, w - 1, -1):
            dp[cap] = max(dp[cap], dp[cap - w] + v)
    return dp[W]

# longest increasing subsequence - O(n log n) with bisect (patience sorting)
def lis(A):
    tails = []
    for x in A:
        i = bisect.bisect_left(tails, x)
        if i == len(tails):
            tails.append(x)
        else:
            tails[i] = x
    return len(tails)

# edit distance (Levenshtein) - O(len(a) * len(b)), two rows
def edit_distance(a, b):
    prev = list(range(len(b) + 1))
    for i, ca in enumerate(a, 1):
        cur = [i]
        for j, cb in enumerate(b, 1):
            cur.append(min(prev[j] + 1, cur[-1] + 1, prev[j - 1] + (ca != cb)))
        prev = cur
    return prev[-1]

assert knapsack([1, 3, 4, 5], [1, 4, 5, 7], 7) == 9
assert lis([10, 9, 2, 5, 3, 7, 101, 18]) == 4
assert edit_distance("kitten", "sitting") == 3

### Memoisation vs tabulation
* **Top-down** (recursion + `functools.lru_cache`): easy to write, only computes needed states, limited by recursion depth.
* **Bottom-up** (loops): no recursion limit, usually faster, allows memory reduction.

## Self-check
1. Why is `(beg + end) / 2` a bug in Python 3, and what is the invariant of `binary_search_le`?
2. What property of the problem makes "binary search on the answer" valid?
3. Prove that the caterpillar does O(n) pointer moves although it has a nested `while`.
4. Give coins for which greedy fails. How would DP handle them?
5. Why does the order of the two loops in the frog/coin DP change the meaning (permutations vs combinations)?